# Considering Linear Regression Models for cross-sectional return predictions



## Objective

Use the features constructed in Notebook 02 to predict next-day cross-sectional equity returns.

The modeling sequence is:

1. Zero-return benchmark
2. Ordinary Least Squares (OLS)
3. Ridge regression
4. Elastic Net

The goals are to study:

- whether the features contain multivariate predictive information,
- OLS coefficient interpretation,
- the effect of multicollinearity,
- coefficient shrinkage under Ridge,
- sparsity under Elastic Net,
- train vs validation performance,
- out-of-sample R²,
- cross-sectional Pearson IC,
- cross-sectional Rank IC.

Model fitting uses the training period (2010–2018).

Hyperparameters are selected using the validation period (2019–2021).

The test period (2022 onward) is not used in this notebook.

Notebook 02 gave us features:
$$ X_{i,t} = (ret1, mom5, mom20,...)$$
Now we try to estimate the returns as a function of the feautres
$$ \hat{y}_{i, t+1} = f(X_{i,t})$$
where $$ y_{i,t+1} = r_{i,t+1} - \bar{r}_{t+1}$$
We take $f$ to be a linear function of the features:
$$ \hat{y} = \beta_0 + \beta_1 x_1 + \beta_2 x_2 + \ldots + \beta_9 x_9 $$

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

import plotly.express as px
import plotly.graph_objects as go

from sklearn.linear_model import (
    LinearRegression,
    Ridge,
    ElasticNet,
)

from sklearn.metrics import mean_squared_error

import json  # To save the chosen hyperparameters at the end

In [2]:
# Load the final feature dataset saved at the end of notebool 2

PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data"

df = pd.read_csv(DATA_DIR / "alpha_dataset.csv", parse_dates=["date"])

df.head()

,date,adj_close,close,high,low,open,volume,ticker,ret_1d,next_obs_date,...,beta_60,ret_1d_z,mom_5_z,mom_20_z,mom_60_z,vol_20_z,vol_60_z,price_ma20_gap_z,dollar_volume_z20_z,beta_60_z
0,2010-03-31,7.028762,8.392857,8.450357,8.373571,8.410357,430659600,AAPL,-0.003604,2010-04-01,...,1.497638,0.072730,1.577789,1.472385,0.625248,0.258562,0.577992,1.687527,-0.201478,1.199480
1,2010-03-31,39.938347,59.840000,60.220001,59.689999,60.000000,4975100,AMGN,-0.004161,2010-04-01,...,0.439355,-0.004092,0.822145,-0.028271,-0.077188,-0.257272,-0.873828,0.610112,-0.178906,-1.426402
2,2010-03-31,56.966679,72.610001,73.059998,71.949997,72.919998,5884600,BA,-0.012512,2010-04-01,...,1.289499,-1.156940,0.229550,1.556566,2.951781,0.651547,0.829979,1.036881,-0.109082,0.683032
3,2010-03-31,13.634262,17.850000,17.980000,17.670000,17.690001,114280900,BAC,0.005067,2010-04-01,...,1.586337,1.269740,1.012569,0.790991,1.086433,0.173629,1.174250,1.504354,-0.966822,1.419567
4,2010-03-31,29.604025,40.500000,41.099998,40.099998,40.799999,55035890,C,-0.009780,2010-04-01,...,1.610091,-0.779761,-1.615686,2.922020,1.691178,3.920722,2.263984,0.335038,0.354980,1.478507


In [3]:
# Define the features are targets clearly

# The nine features constructed in notebook 2
MODEL_FEATURES = [
    "ret_1d_z",
    "mom_5_z",
    "mom_20_z",
    "mom_60_z",
    "vol_20_z",
    "vol_60_z",
    "price_ma20_gap_z",
    "dollar_volume_z20_z",
    "beta_60_z",
]

# Cross-sectional next-day return
TARGET = "target_cs_1d"

In [4]:
missing_cols = set(MODEL_FEATURES + [TARGET]) - set(df.columns)

# assert prevents us from accidentally running the notebook with a malformed dataset
assert not missing_cols, f"Missing required columns: {missing_cols}"

## Split the data into train, validation, and test
* Train learns coefficients of the model
* Validation choses hyperparameters
* Test simulates truly unseen future data. 
Note: Test is created to simply verify the split. We won't evaluate models on it in this notebook.

In [5]:
train = df[df["split"] == "train"].sort_values(["date", "ticker"]).reset_index(drop=True)

validation = df[df["split"] == "validation"].sort_values(["date", "ticker"]).reset_index(drop=True)

test = df[df["split"] == "test"].sort_values(["date", "ticker"]).reset_index(drop=True)

In [6]:
# Look at the summary

split_summary = pd.DataFrame(
    {
        "n_rows": [
            len(train),
            len(validation),
            len(test),
        ],
        "start": [
            train["date"].min(),
            validation["date"].min(),
            test["date"].min(),
        ],
        "end": [
            train["date"].max(),
            validation["date"].max(),
            test["date"].max(),
        ],
        "n_dates": [
            train["date"].nunique(),
            validation["date"].nunique(),
            test["date"].nunique(),
        ],
    },
    index=[
        "train",
        "validation",
        "test",
    ],
)

split_summary

,n_rows,start,end,n_dates
train,66120,2010-03-31,2018-12-31,2204
validation,22710,2019-01-02,2021-12-31,757
test,34980,2022-01-03,2026-08-26,1166


## Build $X$ and $y$
where $X$ is a matrix
$$ X = \begin{pmatrix} x_{11} & x_{12} & \ldots & x_{19} \\ x_{21} & x_{22} & \ldots & x_{29} \\ \vdots & & & \vdots  \end{pmatrix} $$

with each row of the matrix representing one (data, ticker) obervation and each column representing one particular feature.


In [7]:
X_train = train[MODEL_FEATURES]
y_train = train[TARGET]

X_val = validation[MODEL_FEATURES]
y_val = validation[TARGET]

In [8]:
# Some checks
print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)

print()
print("X_val shape:", X_val.shape)
print("y_val shape:", y_val.shape)

print()
print("Missing training features:", X_train.isna().sum().sum())
print("Missing validation features:", X_val.isna().sum().sum())
# Missing features should be zero because we already removed incomplete rows in notebook 2

X_train shape: (66120, 9)
y_train shape: (66120,)

X_val shape: (22710, 9)
y_val shape: (22710,)

Missing training features: 0
Missing validation features: 0


In [9]:
# Inspect the target data
target_summary = pd.DataFrame(
    {
        "train": y_train.describe(),
        "validation": y_val.describe(),
    }
)

target_summary

,train,validation
count,6.612000e+04,2.271000e+04
mean,6.425718e-19,5.812966e-19
std,1.096335e-02,1.502789e-02
min,-1.539199e-01,-1.637701e-01
25%,-5.417117e-03,-7.019033e-03
50%,-1.081902e-04,-1.889527e-04
75%,5.332165e-03,6.757008e-03
max,1.329852e-01,2.315643e-01


Since we defined $$ y_{i,t+1} = r_{i,t+1} - \bar{r}_{t+1}$$ therefore, the mean across the dataset should be zero as seen above

## Define prediction metrics

In [10]:
def oos_r2(y_true, y_pred):
    """
    Out-of-sample R² relative to a zero-return forecast.

    R²_OOS =
        1 - SSE_model / SSE_zero

    Because the target is cross-sectionally demeaned,
    predicting zero is the natural benchmark.
    """

    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    model_sse = np.sum((y_true - y_pred) ** 2)

    zero_sse = np.sum(y_true**2)

    return 1 - model_sse / zero_sse

If $$ R^2_{OOS} > 0 $$ the model predicts better than simply saying $$ \hat{y} = 0 $$
If $$ R^2_{OOS} < 0 $$ the model is actually worse than predicting nothing.

Because the target is demeaned cross-sectionally each day, the zero forecast is
the natural point-forecast benchmark.

Model predictions are not forcibly demeaned again before computing OOS R².
Therefore this metric penalizes both relative-return forecast error and any
common prediction-level error produced by the model.

Rank IC is invariant to common shifts in predictions and remains the primary
metric for the cross-sectional ranking problem.

In [11]:
# Daily cross-sectional IC function
def daily_ic(
    df,
    prediction_col,
    target_col=TARGET,
    method="spearman",
):
    """
    Compute the cross-sectional correlation between
    predictions and realized future returns separately
    for each date.

    method='pearson':
        linear correlation

    method='spearman':
        rank correlation
    """

    dates = []
    ic_values = []

    for date, d in df.groupby("date"):
        valid = d[[prediction_col, target_col]].dropna()

        # Require enough stocks to calculate
        # a meaningful cross-sectional correlation.
        if len(valid) < 5:
            continue

        # Correlation is undefined if either
        # variable is constant.
        if valid[prediction_col].nunique() < 2 or valid[target_col].nunique() < 2:
            ic = np.nan

        else:
            ic = valid[prediction_col].corr(
                valid[target_col],
                method=method,
            )

        dates.append(date)
        ic_values.append(ic)

    return pd.Series(
        ic_values,
        index=pd.to_datetime(dates),
        name=f"{method}_ic",
    )

In [12]:
# General evaluation function
def evaluate_predictions(
    data,
    predictions,
    model_name,
):
    """
    Evaluate one model using both prediction-error
    metrics and cross-sectional alpha metrics.
    """

    temp = data[["date", "ticker", TARGET]].copy()

    temp["prediction"] = predictions

    # Standard prediction metrics
    mse = mean_squared_error(
        temp[TARGET],
        temp["prediction"],
    )

    r2 = oos_r2(
        temp[TARGET],
        temp["prediction"],
    )

    # Cross-sectional Pearson IC
    pearson_ic = daily_ic(
        temp,
        prediction_col="prediction",
        method="pearson",
    )

    # Cross-sectional Rank IC
    rank_ic = daily_ic(
        temp,
        prediction_col="prediction",
        method="spearman",
    )

    return {
        "model": model_name,
        "mse": mse,
        "oos_r2": r2,
        "mean_pearson_ic": pearson_ic.mean(),
        "mean_rank_ic": rank_ic.mean(),
        "rank_ic_std": rank_ic.std(),
        "pct_rank_ic_positive": (rank_ic > 0).mean(),
    }

We use several metrics because a return model can be useful even if it isn't excellent at predicting exact magnitudes
* MSE asks: How close are the numerical forecasts?
* $R^2$ asks: Did we beat predicting zero?
* Pearson IC asks: Do larger predictions correspond to larger returns?
* Rank IC asks: Did we at least get the ordering of stocks right?
For our long-short strategy, Rank IC is especially important.

### Zero forecast baseline.
Before OLS, establish the simplest possible forecast: $$ \hat{y} = 0 $$



In [13]:
zero_train_pred = np.zeros(len(train))

zero_val_pred = np.zeros(len(validation))

print("Training zero-forecast MSE:", mean_squared_error(y_train, zero_train_pred))

print("Validation zero-forecast MSE:", mean_squared_error(y_val, zero_val_pred))

Training zero-forecast MSE: 0.00012019330549570171
Validation zero-forecast MSE: 0.00022582768169965143


The OOS $ R^2 $ of this model is, by definition, 0.


---
## Fit OLS

OLS chooses coefficients that minimize:
$$ \sum_{i} ( y_i - \hat{y}_i)^2 $$

In matrix notation we have:
$$ \hat{\beta} = ( X^T X)^{-1} X^T y $$


In [14]:
ols = LinearRegression()

ols.fit(X_train, y_train)

ols_train_pred = ols.predict(X_train)

ols_val_pred = ols.predict(X_val)

In [15]:
# Look at the OLS intercept and coefficients
print("OLS intercept:", ols.intercept_)


OLS intercept: 6.431117841123062e-19


In [16]:
ols_coef = pd.DataFrame(
    {
        "feature": MODEL_FEATURES,
        "ols_coef": ols.coef_,
    }
)

ols_coef["abs_coef"] = ols_coef["ols_coef"].abs()

ols_coef = ols_coef.sort_values("abs_coef", ascending=False).reset_index(drop=True)

ols_coef

,feature,ols_coef,abs_coef
0,mom_20_z,0.000167,0.000167
1,price_ma20_gap_z,-0.000164,0.000164
2,vol_60_z,0.000110,0.000110
3,beta_60_z,-0.000076,0.000076
4,mom_5_z,-0.000074,0.000074
5,vol_20_z,-0.000042,0.000042
6,mom_60_z,0.000041,0.000041
7,ret_1d_z,-0.000024,0.000024
8,dollar_volume_z20_z,-0.000008,0.000008


In [17]:
fig = px.bar(ols_coef.sort_values("ols_coef"), x="ols_coef", y="feature", orientation="h", title="OLS Coefficients")

fig.show()

The plot above gives the model's estimated direction:
* Positive coefficient --> high feature values predict relative outperformance
* Negative coefficient --> high feature values predict relative underperformance

In [18]:
# Evaluate the OLS
ols_train_metrics = evaluate_predictions(train, ols_train_pred, "OLS - Train")

ols_val_metrics = evaluate_predictions(validation, ols_val_pred, "OLS - Validation")

pd.DataFrame([ols_train_metrics, ols_val_metrics])

,model,mse,oos_r2,mean_pearson_ic,mean_rank_ic,rank_ic_std,pct_rank_ic_positive
0,OLS - Train,0.000120,0.000231,0.014480,0.017660,0.250926,0.529946
1,OLS - Validation,0.000226,-0.000175,-0.001229,0.002137,0.288068,0.513871


## Examining multicollinearity with Variance Inflation Factor (VIF)

For a feature $j$, regress that feature on all the other predictors and calculate:
$$ VIF_j = \frac{1}{1 - R^2_j} $$
If the other variables explain a large fraction of feature $j$, then $$ R^2_j \rightarrow 1 $$ and therefore $$ VIF_j \rightarrow \infty $$

Rough intuition:
* VIF $\approx$ 1 : little multicollinearity
* VIF = 2-5 : noticeable
* VIF > 5 : substantial
* VIF > 10 : often considered severe

In [19]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

vif_table = pd.DataFrame({"feature": MODEL_FEATURES})

vif_table["VIF"] = [variance_inflation_factor(X_train.values, i) for i in range(X_train.shape[1])]

vif_table.sort_values("VIF", ascending=False)


,feature,VIF
6,price_ma20_gap_z,7.282368
2,mom_20_z,4.728478
5,vol_60_z,3.417288
1,mom_5_z,2.741527
4,vol_20_z,2.504450
8,beta_60_z,2.206180
3,mom_60_z,1.463623
0,ret_1d_z,1.230236
7,dollar_volume_z20_z,1.008201


---


## Ridge Regression

Ridge solves: $$ \underset{\beta}{\min} \left[ \sum_{i} ( y_i - X_i \beta)^2 + \alpha \sum_{i} \beta^2_j \right] $$

When: $$ \alpha = 0 $$ Ridge becomes OLS.
As: $$ \alpha \uparrow $$ large coefficients become increasingly expensive.

 So correlated features cannot freely receive large opposing coefficients. This introduces some bias but can reduce variance.

In [20]:
# Some candidate values for alpha
RIDGE_ALPHAS = [0.01, 0.1, 1, 10, 100, 1_000, 10_000, 30_000, 100_000, 300_000, 1_000_000]


### Tune Ridge using validation Rank IC

We train models with various values of $\alpha$ on the same training data from $$2010-2018$$
 and then we ask how each value performs during $$2019-2021$$

Our eventual decision is: $$ \text{rank stocks} \rightarrow \text{long high scores} \rightarrow \text{short low scores} $$
Therefore, correct relative ordering is economically more important than exact return magnitude.
Since our eventual strategy ranks stocks, we'll use validation Rank IC as the primary selection metric.

In [21]:
ridge_results = []
ridge_models = {}

for alpha in RIDGE_ALPHAS:
    model = Ridge(alpha=alpha)

    # Fit only on training data
    model.fit(X_train, y_train)

    # Predict validation period
    val_pred = model.predict(X_val)

    metrics = evaluate_predictions(validation, val_pred, model_name=f"Ridge alpha={alpha}")

    metrics["alpha"] = alpha

    ridge_results.append(metrics)

    ridge_models[alpha] = model


ridge_results = pd.DataFrame(ridge_results)

ridge_results

,model,mse,oos_r2,mean_pearson_ic,mean_rank_ic,rank_ic_std,pct_rank_ic_positive,alpha
0,Ridge alpha=0.01,0.000226,-0.000175,-0.001229,0.002137,0.288068,0.513871,0.01
1,Ridge alpha=0.1,0.000226,-0.000175,-0.001229,0.002137,0.288068,0.513871,0.10
2,Ridge alpha=1,0.000226,-0.000175,-0.001229,0.002150,0.288086,0.513871,1.00
3,Ridge alpha=10,0.000226,-0.000175,-0.001228,0.002127,0.288198,0.513871,10.00
4,Ridge alpha=100,0.000226,-0.000175,-0.001228,0.001870,0.288298,0.512550,100.00
5,Ridge alpha=1000,0.000226,-0.000170,-0.001212,0.002223,0.290179,0.513871,1000.00
6,Ridge alpha=10000,0.000226,-0.000140,-0.000868,0.005567,0.299239,0.531044,10000.00
7,Ridge alpha=30000,0.000226,-0.000100,-0.000302,0.008357,0.303407,0.525760,30000.00
8,Ridge alpha=100000,0.000226,-0.000048,0.000194,0.008657,0.307274,0.519155,100000.00
9,Ridge alpha=300000,0.000226,-0.000018,0.000134,0.009320,0.310012,0.520476,300000.00


In [22]:
# select the best Ridge alpha
ridge_results = ridge_results.sort_values("mean_rank_ic", ascending=False).reset_index(drop=True)

ridge_results

,model,mse,oos_r2,mean_pearson_ic,mean_rank_ic,rank_ic_std,pct_rank_ic_positive,alpha
0,Ridge alpha=300000,0.000226,-0.000018,0.000134,0.009320,0.310012,0.520476,300000.00
1,Ridge alpha=1000000,0.000226,-0.000005,-0.000035,0.008890,0.311513,0.509908,1000000.00
2,Ridge alpha=100000,0.000226,-0.000048,0.000194,0.008657,0.307274,0.519155,100000.00
3,Ridge alpha=30000,0.000226,-0.000100,-0.000302,0.008357,0.303407,0.525760,30000.00
4,Ridge alpha=10000,0.000226,-0.000140,-0.000868,0.005567,0.299239,0.531044,10000.00
5,Ridge alpha=1000,0.000226,-0.000170,-0.001212,0.002223,0.290179,0.513871,1000.00
6,Ridge alpha=1,0.000226,-0.000175,-0.001229,0.002150,0.288086,0.513871,1.00
7,Ridge alpha=0.1,0.000226,-0.000175,-0.001229,0.002137,0.288068,0.513871,0.10
8,Ridge alpha=0.01,0.000226,-0.000175,-0.001229,0.002137,0.288068,0.513871,0.01
9,Ridge alpha=10,0.000226,-0.000175,-0.001228,0.002127,0.288198,0.513871,10.00


In [23]:
# Check how much ridge is shrinking the predictions
for alpha in [10_000, 30_000, 100_000, 300_000, 1_000_000]:
    model = ridge_models[alpha]
    pred = model.predict(X_val)

    print(alpha, "prediction std =", np.std(pred), "coef norm =", np.linalg.norm(model.coef_))

10000 prediction std = 0.00013086074857762703 coef norm = 0.00016765378322519336
30000 prediction std = 9.993840751588955e-05 coef norm = 0.0001114565282865341
100000 prediction std = 5.9528784111570246e-05 coef norm = 5.8369008762992444e-05
300000 prediction std = 2.9485304575120495e-05 coef norm = 2.6355360628504043e-05
1000000 prediction std = 1.1018224604580501e-05 coef norm = 9.282930498439188e-06


In [24]:
best_ridge_alpha = ridge_results.loc[0, "alpha"]

print("Best Ridge alpha:", best_ridge_alpha)

Best Ridge alpha: 300000.0


In [25]:
# Since we have already fitted the models with various values of alpha above, we just select the one with best results

best_ridge = ridge_models[best_ridge_alpha]

ridge_train_pred = best_ridge.predict(X_train)

ridge_val_pred = best_ridge.predict(X_val)

In [26]:
ridge_train_metrics = evaluate_predictions(
    train,
    ridge_train_pred,
    "Ridge - Train",
)

ridge_val_metrics = evaluate_predictions(
    validation,
    ridge_val_pred,
    "Ridge - Validation",
)

pd.DataFrame(
    [
        ridge_train_metrics,
        ridge_val_metrics,
    ]
)

,model,mse,oos_r2,mean_pearson_ic,mean_rank_ic,rank_ic_std,pct_rank_ic_positive
0,Ridge - Train,0.000120,0.000060,0.013051,0.01860,0.264972,0.529946
1,Ridge - Validation,0.000226,-0.000018,0.000134,0.00932,0.310012,0.520476


### Compare OLS and Ridge coefficients

Ridge usually should produce: $$ | \beta^{Ridge}_j | < | \beta^{OLS}_j | $$
This represents Shrinkage

In [27]:
coef_compare = pd.DataFrame(
    {
        "feature": MODEL_FEATURES,
        "OLS": ols.coef_,
        "Ridge": best_ridge.coef_,
    }
)

coef_compare

,feature,OLS,Ridge
0,ret_1d_z,-0.000024,-0.000011
1,mom_5_z,-0.000074,-0.000019
2,mom_20_z,0.000167,0.000005
3,mom_60_z,0.000041,0.000007
4,vol_20_z,-0.000042,-0.000001
5,vol_60_z,0.000110,0.000004
6,price_ma20_gap_z,-0.000164,-0.000010
7,dollar_volume_z20_z,-0.000008,-0.000003
8,beta_60_z,-0.000076,-0.000004


In [28]:
# Plot
coef_long = coef_compare.melt(
    id_vars="feature",
    var_name="model",
    value_name="coefficient",
)

fig = px.bar(
    coef_long,
    x="feature",
    y="coefficient",
    color="model",
    barmode="group",
    title=("OLS vs Ridge Coefficients"),
)

fig.show()

### Ridge regularization path

This gives a visual representation of what Ridge is actually doing.

As $\alpha$ increases (left to right), the coefficients move towards zero. Ridge does not make coefficients exactly zero, it shrinks them.

In [29]:
ridge_path = []

for alpha in RIDGE_ALPHAS:
    model = ridge_models[alpha]

    for feature, coef in zip(
        MODEL_FEATURES,
        model.coef_,
    ):
        ridge_path.append(
            {
                "alpha": alpha,
                "feature": feature,
                "coefficient": coef,
            }
        )


ridge_path = pd.DataFrame(ridge_path)

In [30]:
fig = px.line(
    ridge_path,
    x="alpha",
    y="coefficient",
    color="feature",
    log_x=True,
    markers=True,
    title="Ridge Coefficient Shrinkage Path",
)

fig.show()

---

## Elastic Net

Elastic Net combines both L1 and L2 regularizations, i.e., $$ \text{Elastic Net} = \text{Lasso} + \text{Ridge} $$

The L1 component can force some coefficients to exactly 0. So Elastic Net can perform feature selection

In [31]:
# Define a small grid of hyperparameters
EN_ALPHAS = [1e-6, 3e-6, 1e-5, 3e-5, 1e-4, 3e-4, 1e-3]

EN_L1_RATIOS = [0.1, 0.5, 0.9]
# If L1_Ratio is near 0 --> mostly Ridge-like
# If L1_Ratio is near 1 --> mostly Lasso-like

### Tune Elastic Net and select the best one

In [32]:
elastic_results = []
elastic_models = {}

for alpha in EN_ALPHAS:
    for l1_ratio in EN_L1_RATIOS:
        model = ElasticNet(
            alpha=alpha,
            l1_ratio=l1_ratio,
            max_iter=100_000,
        )

        # Fit using training data only
        model.fit(X_train, y_train)

        # Evaluate on validation
        val_pred = model.predict(X_val)

        metrics = evaluate_predictions(
            validation,
            val_pred,
            model_name=(f"ElasticNet alpha={alpha}, l1_ratio={l1_ratio}"),
        )

        metrics["alpha"] = alpha
        metrics["l1_ratio"] = l1_ratio

        elastic_results.append(metrics)

        elastic_models[(alpha, l1_ratio)] = model

In [33]:
# Turn into a table
elastic_results = pd.DataFrame(elastic_results)

elastic_results = elastic_results.sort_values("mean_rank_ic", ascending=False).reset_index(drop=True)

elastic_results

,model,mse,oos_r2,mean_pearson_ic,mean_rank_ic,rank_ic_std,pct_rank_ic_positive,alpha,l1_ratio
0,"ElasticNet alpha=3e-05, l1_ratio=0.9",0.000226,-0.000101,-0.002767,0.008628,0.309960,0.517834,0.000030,0.9
1,"ElasticNet alpha=0.0003, l1_ratio=0.1",0.000226,-0.000094,-0.003029,0.007714,0.311907,0.515192,0.000300,0.1
2,"ElasticNet alpha=3e-05, l1_ratio=0.5",0.000226,-0.000124,-0.002427,0.006399,0.303501,0.519155,0.000030,0.5
3,"ElasticNet alpha=0.0001, l1_ratio=0.1",0.000226,-0.000133,-0.002504,0.005050,0.297911,0.520476,0.000100,0.1
4,"ElasticNet alpha=1e-05, l1_ratio=0.9",0.000226,-0.000136,-0.002389,0.004393,0.297656,0.519155,0.000010,0.9
5,"ElasticNet alpha=0.0001, l1_ratio=0.5",0.000226,-0.000041,-0.004505,0.004208,0.317930,0.511229,0.000100,0.5
6,"ElasticNet alpha=0.001, l1_ratio=0.1",0.000226,-0.000005,-0.004505,0.004208,0.317930,0.511229,0.001000,0.1
7,"ElasticNet alpha=0.0001, l1_ratio=0.9",0.000226,-0.000010,-0.004505,0.004208,0.317930,0.511229,0.000100,0.9
8,"ElasticNet alpha=1e-05, l1_ratio=0.5",0.000226,-0.000151,-0.001872,0.003246,0.292086,0.519155,0.000010,0.5
9,"ElasticNet alpha=3e-05, l1_ratio=0.1",0.000226,-0.000160,-0.001606,0.002316,0.290850,0.517834,0.000030,0.1


In [34]:
# Select the best Elastic Net
best_en_alpha = elastic_results.loc[0, "alpha"]

best_en_l1_ratio = elastic_results.loc[0, "l1_ratio"]

print("Best Elastic Net alpha:", best_en_alpha)

print("Best Elastic Net l1_ratio:", best_en_l1_ratio)

Best Elastic Net alpha: 3e-05
Best Elastic Net l1_ratio: 0.9


In [35]:
# Retrive the model
best_elastic = elastic_models[(best_en_alpha, best_en_l1_ratio)]

In [36]:
# Predictions
elastic_train_pred = best_elastic.predict(X_train)

elastic_val_pred = best_elastic.predict(X_val)

In [37]:
# Inspect Elastic Net coefficients
elastic_coef = pd.DataFrame(
    {
        "feature": MODEL_FEATURES,
        "coefficient": best_elastic.coef_,
    }
)

elastic_coef["selected"] = elastic_coef["coefficient"] != 0

elastic_coef.sort_values("coefficient")

,feature,coefficient,selected
1,mom_5_z,-0.000097,True
0,ret_1d_z,-0.000012,True
5,vol_60_z,0.000000,False
4,vol_20_z,-0.000000,False
6,price_ma20_gap_z,-0.000000,False
7,dollar_volume_z20_z,-0.000000,False
8,beta_60_z,-0.000000,False
3,mom_60_z,0.000021,True
2,mom_20_z,0.000022,True


If the Elastic Net coefficient is zero, it means the model decided some predictors did not contribute enough incremental information to justify retianing their coefficients.

Note: A zero coefficient doesn't prove that the variable/feature is economically meaningless. It means that given the other predictors and this regularization strength, Elastic Net chose not to use it.

## Final linear-model comparison

In [38]:
# Let's just build one table
results = pd.DataFrame(
    [
        ols_train_metrics,
        ols_val_metrics,
        ridge_train_metrics,
        ridge_val_metrics,
        evaluate_predictions(
            train,
            elastic_train_pred,
            "Elastic Net - Train",
        ),
        evaluate_predictions(
            validation,
            elastic_val_pred,
            "Elastic Net - Validation",
        ),
    ]
)

results

,model,mse,oos_r2,mean_pearson_ic,mean_rank_ic,rank_ic_std,pct_rank_ic_positive
0,OLS - Train,0.000120,0.000231,0.014480,0.017660,0.250926,0.529946
1,OLS - Validation,0.000226,-0.000175,-0.001229,0.002137,0.288068,0.513871
2,Ridge - Train,0.000120,0.000060,0.013051,0.018600,0.264972,0.529946
3,Ridge - Validation,0.000226,-0.000018,0.000134,0.009320,0.310012,0.520476
4,Elastic Net - Train,0.000120,0.000138,0.010702,0.015046,0.265839,0.527223
5,Elastic Net - Validation,0.000226,-0.000101,-0.002767,0.008628,0.309960,0.517834


In [39]:
# For easier comparison
validation_results = results[results["model"].str.contains("Validation")].copy()

validation_results.sort_values("mean_rank_ic", ascending=False)

,model,mse,oos_r2,mean_pearson_ic,mean_rank_ic,rank_ic_std,pct_rank_ic_positive
3,Ridge - Validation,0.000226,-0.000018,0.000134,0.009320,0.310012,0.520476
5,Elastic Net - Validation,0.000226,-0.000101,-0.002767,0.008628,0.309960,0.517834
1,OLS - Validation,0.000226,-0.000175,-0.001229,0.002137,0.288068,0.513871


## Store validation predictions

In [40]:
validation_predictions = validation[
    [
        "date",
        "ticker",
        "fwd_ret_1d",
        "target_cs_1d",
    ]
].copy()

validation_predictions["ols_pred"] = ols_val_pred

validation_predictions["ridge_pred"] = ridge_val_pred

validation_predictions["elastic_net_pred"] = elastic_val_pred

In [41]:
validation_predictions.head()

,date,ticker,fwd_ret_1d,target_cs_1d,ols_pred,ridge_pred,elastic_net_pred
0,2019-01-02,AAPL,-0.099608,-0.075324,-0.000052,-0.000011,-0.000081
1,2019-01-02,AMGN,-0.015216,0.009068,-0.000090,0.000003,0.000005
2,2019-01-02,BA,-0.039869,-0.015585,-0.000191,-0.000038,-0.000143
3,2019-01-02,BAC,-0.016026,0.008258,-0.000499,-0.000054,-0.000169
4,2019-01-02,C,-0.018120,0.006164,-0.000489,-0.000054,-0.000179


### Compare daily Rank IC through time

In [42]:
model_prediction_cols = [
    "ols_pred",
    "ridge_pred",
    "elastic_net_pred",
]

rank_ic_series = []

for pred_col in model_prediction_cols:
    ic = daily_ic(
        validation_predictions,
        prediction_col=pred_col,
        method="spearman",
    )

    temp = pd.DataFrame(
        {
            "date": ic.index,
            "rank_ic": ic.values,
            "model": pred_col,
        }
    )

    rank_ic_series.append(temp)


rank_ic_series = pd.concat(rank_ic_series, ignore_index=True)

In [43]:
# Because daily IC is extemely noisy, plot a 60-day rolling average
rank_ic_series = rank_ic_series.sort_values(["model", "date"])

rank_ic_series["rank_ic_60d"] = rank_ic_series.groupby("model")["rank_ic"].transform(
    lambda s: s.rolling(60, min_periods=20).mean()
)

In [44]:
fig = px.line(
    rank_ic_series,
    x="date",
    y="rank_ic_60d",
    color="model",
    title=("Validation-Period 60-Day Rolling Rank IC"),
)

fig.add_hline(y=0, line_dash="dash")

fig.show()

The average Rank IC alone might be, say 0.015, but we want to know whether that came from:
- steady tiny signal, or
- huge positive signal for six months and then nothing for two years.

Alpha stability matters

## Save validation predictions and selected hyperparameters

In [45]:
validation_predictions.to_csv(
    DATA_DIR / "linear_model_validation_predictions.csv",
    index=False,
)

In [46]:
selected_params = {
    "ridge": {"alpha": float(best_ridge_alpha)},
    "elastic_net": {
        "alpha": float(best_en_alpha),
        "l1_ratio": float(best_en_l1_ratio),
    },
}

In [47]:
with open(DATA_DIR / "linear_model_params.json", "w") as f:
    json.dump(
        selected_params,
        f,
        indent=4,
    )

In [48]:
selected_params

{'ridge': {'alpha': 300000.0},
 'elastic_net': {'alpha': 3e-05, 'l1_ratio': 0.9}}

#### We've now made the hyperparameter decisions using $$ 2019 - 2021 $$ and now we freeze them
#### Later, when we run the 2022+ walk-forward experiment, we would not tune these values again using the test set.

## Initial Linear-Model Findings

The predictors exhibit substantial multicollinearity, particularly between
20-day momentum and distance from the 20-day moving average.

OLS shows a small in-sample ranking relationship, while its validation
performance is much weaker. Ridge and Elastic Net produce higher observed
validation Rank IC than OLS, suggesting that regularization may be useful
in this noisy, correlated predictor set.

However, the differences in mean validation Rank IC are small relative to
the day-to-day variability of IC. These results should therefore be treated
as descriptive model-selection evidence rather than proof that regularization
provides a statistically significant improvement.

Among the linear specifications tested, Ridge achieved the highest validation
Rank IC at alpha = 300,000. Elastic Net selected alpha = 3e-5 and
l1_ratio = 0.9.

Because l1_ratio = 0.9 is the upper edge of the pre-specified Elastic Net
grid, the result should be interpreted as favoring relatively strong L1
regularization, not as establishing 0.9 as an optimal value.

All linear models have validation OOS R² close to zero, indicating that exact
next-day return magnitudes are extremely difficult to forecast. Rank-based
metrics remain more relevant to the intended cross-sectional portfolio.

Formal comparison with the nonlinear model and HAC inference on daily Rank IC
are performed in Notebook 04.